# 原 Stage1 输入与固定 0bp 组成账户：全季度只读归因

冻结 Stage1 的39项输入与同一个已完成的候选账户逐日对齐，保留944个资金日和16个完整/部分季度。2024Q3与2026Q3为既有失败归因中的好/差防守示例，只作解释；不训练模型、不重放账户、不设交易开关。

In [1]:
import torch
import json,hashlib,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder,_ssh
from src.model.minute_pit_allocator import _scaled
root=Path.cwd();card=root/'research/stage1_beta_composition_original_inputs_attribution_20261022';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
original_spec=plan['source_original'];account_spec=plan['source_account'];original_card=root/original_spec['card'];account_card=root/account_spec['card']
assert sha(root/original_spec['provider']/'qlib_store_manifest.json')==original_spec['provider_manifest_sha256']
assert sha(account_card/'summary.json')==account_spec['summary_sha256']
for spec,name,key in [(original_spec,'features_artifact','features_sha256'),(original_spec,'model_artifact','model_sha256')]:
 path=original_card/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts'/spec[name]
 assert sha(path)==spec[key]
for name,key in [('native_artifact','native_artifact_sha256'),('quarter_artifact','quarter_artifact_sha256')]:
 path=account_card/'qlib_recorder'/account_spec['experiment_id']/account_spec['recorder_id']/'artifacts'/account_spec[name]
 assert sha(path)==account_spec[key]
qlib.init(provider_uri=str(root/original_spec['provider']),region='us',kernels=1)
with R.uri_context(str(original_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=original_spec['recorder_id'],experiment_id=original_spec['experiment_id'])
 original=rec.load_object(original_spec['model_artifact'])
 features=rec.load_object(original_spec['features_artifact'])
with R.uri_context(str(account_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=account_spec['recorder_id'],experiment_id=account_spec['experiment_id'])
 book=rec.load_object(account_spec['native_artifact'])
 quarter=rec.load_object(account_spec['quarter_artifact'])
authority=book['authority'].copy().reset_index(drop=True)
targets=book['targets'].copy().reset_index(drop=True)
signals=pd.DatetimeIndex(pd.to_datetime(authority.signal_date))
assert len(authority)==plan['required_complete_rows'] and len(targets)==len(authority)
assert signals.is_monotonic_increasing and not signals.has_duplicates
pd.testing.assert_index_equal(signals,pd.DatetimeIndex(pd.to_datetime(targets.signal_date)),check_names=False)
assert signals.isin(features.index).all()
base_names=list(original.base_features)
assert len(base_names)==39 and len(quarter)==plan['required_quarters']
primary=_scaled(features.loc[signals,base_names],base_names,original.primary_scaler)
assert torch.isfinite(primary).all()
scaled=pd.DataFrame(primary.numpy(),columns=base_names)
daily=authority[['signal_date','trade_date','risk_state','execute_open_target','strategy_return',
 'ending_soxl_weight','ending_soxx_weight','ending_cash_weight']].copy()
for name in ('soxl_weight','soxx_weight','cash_weight'):
 daily['target_'+name]=targets[name].to_numpy(float)
for name in base_names:daily['z_'+name]=scaled[name].to_numpy(float)
daily['period']=pd.to_datetime(daily.trade_date).dt.to_period('Q').astype(str)
assert len(daily.period.unique())==plan['required_quarters']
q=quarter.copy();assert set(q.period)==set(daily.period.unique())
feature_means=daily.groupby('period',sort=True)[['z_'+name for name in base_names]].mean()
feature_medians=daily.groupby('period',sort=True)[['z_'+name for name in base_names]].median()
account_rows=[]
for period,group in daily.groupby('period',sort=True):
 saved=q.loc[q.period.eq(period)].iloc[0]
 account_rows.append({'period':period,'sessions':len(group),'complete_period':bool(saved.complete_period),
   'SOXX_relative_wealth':float(saved.relative_wealth),
   'mean_target_soxl_weight':float(group.target_soxl_weight.mean()),
   'mean_target_cash_weight':float(group.target_cash_weight.mean()),
   'mean_ending_soxl_weight':float(group.ending_soxl_weight.mean()),
   'mean_ending_cash_weight':float(group.ending_cash_weight.mean()),
   'target_to_ending_cash_gap':float(group.ending_cash_weight.mean()-group.target_cash_weight.mean()),
   'risk_state_ge2_fraction':float(group.risk_state.ge(2).mean()),
   'hold_fraction':float((~group.execute_open_target).mean())})
all_quarters=pd.DataFrame(account_rows)
a,b=plan['illustrative_periods']
contrast={name:float(feature_means.loc[b,'z_'+name]-feature_means.loc[a,'z_'+name]) for name in base_names}
sorted_names=sorted(contrast,key=lambda name:abs(contrast[name]),reverse=True)
contrast_top=[{'original_Stage1_feature':name,'mean_standardized_'+b+'_minus_'+a:contrast[name],
               a+'_mean_z':float(feature_means.loc[a,'z_'+name]),b+'_mean_z':float(feature_means.loc[b,'z_'+name])}
              for name in sorted_names[:8]]
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path='inputs',plan=plan,source_account_identity=account_spec,source_original_identity=original_spec)
 recorder.save_objects(artifact_path='analysis/daily',joined=daily)
 recorder.save_objects(artifact_path='analysis/quarter',features_mean_z=feature_means,features_median_z=feature_medians,
                       all_account_quarters=all_quarters,all_feature_contrasts=contrast)
 recorder.log_metrics(matched_sessions=len(daily),quarters=len(all_quarters),new_fits=0,new_accounts=0)
 rec_id=recorder.id;exp_id=str(recorder.experiment_id)
summary={'experiment':plan['experiment'],'decision':'descriptive_original_Stage1_input_regime_attribution_only',
 'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_recorder_ids':{'original':original_spec['recorder_id'],'account':account_spec['recorder_id']},
 'matched_sessions':len(daily),'quarters':len(all_quarters),'original_features':len(base_names),
 'all_quarter_account_summary':account_rows,'illustrative_feature_contrast_top8':contrast_top,
 'illustrative_periods':plan['illustrative_periods'],'new_model_fits':0,'new_native_accounts':0,'new_bootstrap_runs':0,
 'recorder_id':rec_id,'experiment_id':exp_id,'recorder_uri':str(tracking),
 'artifact_policy':plan['artifact_policy'],
 'limitations':['Illustrative periods were identified from reused outcomes; standardized input contrasts are descriptive, not an ex-ante rule.',
                'Risk state and holdings are account feedback, not independent market states.',
                'No new source, model, economic account or qualified forward evidence.']}
atomic_json(summary,card/'summary.json')
artifacts=tracking/exp_id/rec_id/'artifacts'
atomic_json({'recorder_id':rec_id,'entries':[{'path':str(x.relative_to(card)),'sha256':sha(x),'bytes':x.stat().st_size}
 for x in sorted(artifacts.rglob('*')) if x.is_file()]},card/'verification_manifest.json')
binding=json.loads((account_card/'nas_binding.json').read_text());assert binding.get('verified') is True
binding.pop('evidence',None);binding['research_card']=str(card.relative_to(root));atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec_id,experiment_id=exp_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
{k:summary[k] for k in ('decision','matched_sessions','quarters','original_features','illustrative_feature_contrast_top8')}


{'decision': 'descriptive_original_Stage1_input_regime_attribution_only',
 'matched_sessions': 944,
 'quarters': 16,
 'original_features': 39,
 'illustrative_feature_contrast_top8': [{'original_Stage1_feature': 'rv60',
   'mean_standardized_2026Q3_minus_2024Q3': 1.9573471068470099,
   '2024Q3_mean_z': 1.2139928371580027,
   '2026Q3_mean_z': 3.1713399440050125},
  {'original_Stage1_feature': 'semis_vs_cross_60',
   'mean_standardized_2026Q3_minus_2024Q3': 1.212981137141469,
   '2024Q3_mean_z': -0.940209954205784,
   '2026Q3_mean_z': 0.2727711829356849},
  {'original_Stage1_feature': 'vix_z_60',
   'mean_standardized_2026Q3_minus_2024Q3': -1.0832092947493948,
   '2024Q3_mean_z': 0.6312860732577974,
   '2026Q3_mean_z': -0.45192322149159736},
  {'original_Stage1_feature': 'cross_dispersion_20',
   'mean_standardized_2026Q3_minus_2024Q3': 0.9343490566534456,
   '2024Q3_mean_z': 0.6547173117287457,
   '2026Q3_mean_z': 1.5890663683821913},
  {'original_Stage1_feature': 'soxx_lead_10',
   'mea

## 中文结论

从原生Qlib来源Recorder与已完成的候选账户Recorder对齐全部944资金信号日、16个季度及原39项训练标准化输入。未拟合、未重放、未重抽样，所有日级输入/账户和16×39特征矩阵保存在新Qlib Recorder。

既有示例2024Q3相对SOXX财富 1.0659，平均目标/期末现金 0.392/0.393，高风险状态占 54.7%；2026Q3分别为 0.8919、0.620/0.620、82.8%。两个季度目标到期末现金的均值差都不足0.1个百分点。

原Stage1已收到显著不同的先验输入：2026Q3相对2024Q3，`rv60`训练标准化均值差+1.96，`semis_vs_cross_60` +1.21，`vix_z_60` -1.08，`cross_dispersion_20` +0.93。完整39项和全部季度在Recorder，以上只作事后解释；不能因此把某一差值或风险状态做成交易阈值，也不能据此断言信息已被模型有效使用。

归因指向目标配置/风险状态与后续路径反馈的研究问题；至少在这两季，期末现金差距并非主要由目标到期末的均值漏传造成。费用、逐日整股与动态反馈仍可能影响全期，未做反事实因果识别。下一特征必须在原39输入之外提供决策时可得的恢复/持续走弱信息，先过来源和匹配信息门，再做完整0bp自身账户。已关闭的组成比例、风险上限及季度窗口不调参救援。

本卡决定 `descriptive_original_Stage1_input_regime_attribution_only`，不改变之前0bp失败结论。Recorder `66739637c55a4558818b18f978852e10` FINISHED，NAS归档SHA `fe6afa906c494ed13c12c0d2849594b1173998ec6aba0615354782cb49b7e15f` 已远端回读；0模型拟合、0经济账户。总SOXX目标仍未完成。
